In [ ]:
# !pip install openai

In [ ]:
# convert csv from pandas to sql
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/gitmystuff/Datasets/main/Auto.csv')
print(df.shape)
print(df.info())
df.head()

In [ ]:
# pandas groupby example
df['origin'] = df['origin'].astype('object')
df.groupby('origin')['mpg'].mean()

In [ ]:
# pandas to sql
# what is the average mpg for the origin of cars
# --> SELECT AVG(mpg) FROM table GROUP BY ...
from sqlalchemy import create_engine, text

db = create_engine('sqlite://', echo=True)

In [ ]:
data = df.to_sql(name='MPG', con=db)

In [ ]:
# connect and query
with db.connect() as conn:
  # makes connection, allows queries, and closes
  results = conn.execute(text('SELECT * FROM MPG'))

results.all()

In [ ]:
# connect and query
with db.connect() as conn:
  # makes connection, allows queries, and closes
  results = conn.execute(text('SELECT AVG(mpg) FROM MPG GROUP BY origin'))

results.all()

In [ ]:
import os
import openai

In [ ]:
# Example: reuse your existing OpenAI setup
import re
from openai import OpenAI

def prompt_input():
    nlsql = input('What would you like to know?: ')
    return nlsql

# Point to the local server
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# prompt = 'Using the following table information, ### sqlite SQL table with fields:\n    #\n    # MPG(mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name)\n    #\n    ### write an sql statement for the following: get the average mpg for each origin\nSELECT'
nltext = prompt_input()
prompt = f'Using the following MPG table with fields: (mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name) write an sql statement for the following: {nltext}. Do not use line breaks or tick marks. SELECT'

completion = client.chat.completions.create(
  model="TheBloke/CodeLlama-7B-Instruct-GGUF/codellama-7b-instruct.Q4_K_S.gguf",
  messages=[
    {"role": "system", "content": "You are an AI assitant transforming natural language into SQL."},
    {"role": "user", "content": prompt}
  ],
  temperature = 0,
)

query = completion.choices[0].message.content
print(query)
with db.connect() as conn:
    results = conn.execute(text(query))

print(results.all())
# show me the average mpg for each origin